# Routing and attention lenses

Portable versions of the original cheap/strong routing and 18-lens examples. Routing labels do not dispatch a completion. Relative lane costs are illustrative, not measured savings. All example business figures are fictional fixtures.

In [ ]:
from pathlib import Path
import sys, os
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').exists())
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))
from jev_lab.safety import live_enabled
RUN_LIVE = live_enabled()
print('Live API calls:', RUN_LIVE)
# Live calls send cell inputs to TypeSafe and may incur charges.
# Explicit opt-in: export JEV_LAB_LIVE=1 and TYPESAFE_API_KEY before launching Jupyter.
from typing import Literal
from enum import Enum
from pydantic import BaseModel, Field
import jev


---
# Part II — Cheap decision-making: route the *conversation data*, downgrade cheap follow-ups

The findings on cognitive behaviors (looked up live this session) all point one way: **not every
turn deserves a frontier model** — and in a real conversation most turns don't.

- **CDR — Cognitive Decision Routing** ([arXiv 2508.16636](https://arxiv.org/abs/2508.16636)):
  a Kahneman fast/slow router scores a query's cognitive complexity and routes cheap vs strong.
  Complex reasoning gains; **simple factual queries lose nothing by going cheap** (~34% cost cut).
- **RouteLLM / FrugalGPT cascades** ([tianpan.co](https://tianpan.co/blog/2025/11/03/llm-routing-model-cascades)):
  start cheap, **escalate on low confidence**. Naive self-reported confidence is **poorly
  calibrated** — jev's **Noul** is the fix.
- The behavior the research keeps surfacing: **follow-ups usually need far less power than the
  turn that opened the thread.** "Why?", "and the next one?", "yes, but shorter", "what about the
  second one?" are short, anaphoric, context-bound — they ride the conversation's established
  frame instead of building a new one. Those are exactly the turns to **downgrade to a cheap lane.**

So the routing decision below operates on the **actual conversation data** — the new message,
the accumulated context, and the follow-up structure — not on synthetic one-liner queries.
jev reads that state as one System One call and decides, per turn:

1. **`followup_detected`** — is this turn a follow-up to established context?
2. **`context_sufficient`** — can the existing context mostly answer it (no new heavy reasoning)?
3. **`lane`** — `cheap` (haiku-class) vs `strong` (opus-class) for *this* turn.
4. **confidence gate** — jev's calibrated p(cheap-answers-well) escalates only when it isn't sure.

The payoff is the **follow-up discount**: a thread whose *opener* needs the strong lane can have its
follow-ups served cheaply, which is where most of the tokens actually are.

## 14 · A conversation as data

A thread is a list of turns. Each turn carries the role, the text, and (on the model side) a record
of which lane served it. We build a realistic thread: an opener that genuinely needs reasoning,
then a run of progressively cheaper follow-ups.

In [ ]:
if RUN_LIVE:
    conversation = [
        {"role": "user", "text": "We're a 12-person B2B SaaS startup, ~$40k MRR, 8 months runway. Should we raise a bridge round now or cut burn to extend runway? Consider investor signaling, team morale, and the current funding market."},
        # ^ opener: genuinely multi-factor -> should route STRONG
    ]
    # follow-ups that arrive after the strong answer:
    followups = [
        "Why?",
        "What's the single biggest risk in the raise-now path?",
        "And if we cut burn instead — what's the first thing to cut?",
        "ok. Summarize that as three bullets.",
        "thanks. make the bullets shorter.",
        "What about the second bullet — expand it a little.",
        "Now something new: design a distributed rate limiter that is fair across tenants and degrades gracefully under DDoS, and justify the algorithm choice.",
    ]
    print("thread: 1 opener +", len(followups), "follow-ups")
    for f in followups: print("  ·", f[:70])
else:
    print('Live example skipped; set JEV_LAB_LIVE=1 with your own API key to run.')

## 15 · The router — decides from the message + context + follow-up structure

The jev state **is the conversation data**: the new message, a recap of the context so far, and how
many turns deep we are. From that, jev answers the three routing questions. `bool_threshold` is the
aggressiveness dial for downgrading follow-ups to the cheap lane (raise it → fewer follow-ups stay
cheap → more escalations).

In [ ]:
if RUN_LIVE:
    # Illustrative lane labels, not health-checked provider routes.
    CHEAP_LANE  = "anthropic/claude-haiku-4-5-20251001"   # System 1 — fast, cheap
    STRONG_LANE = "anthropic/claude-opus-4-6"             # System 2 — slow, expensive
    LANE_COST   = {CHEAP_LANE: 1.0, STRONG_LANE: 5.0}     # relative per-turn cost (haiku:opus ~ 1:5)

    class TurnRoute(BaseModel):
        followup_detected: bool = Field(description="Is this new message a follow-up that builds on the conversation so far, rather than a brand-new topic?")
        context_sufficient: bool = Field(description="Can the existing conversation context mostly answer this turn without new heavy reasoning?")
        lane: Literal["cheap", "strong"] = Field(description="cheap: a short/anaphoric follow-up or simple request a fast model handles. strong: new topic or genuinely complex multi-step reasoning.")

    @jev.fn(bool_threshold=0.5)
    def route_turn(new_message: str, context_recap: str, turns_so_far: int) -> TurnRoute:
        """Route this conversation turn to a cheap or strong model lane.

        Conversation so far ({{ turns_so_far }} turns):
        {{ context_recap }}

        New message:
        {{ new_message }}
        """
        return route_turn.state()

    def recap(turns):
        return "\n".join(f"{t['role']}: {t['text'][:90]}" for t in turns) or "(start of conversation)"

    # smoke: the opener should be strong, "Why?" should be cheap
    print(route_turn(conversation[0]["text"], recap([]), 0))
    print(route_turn("Why?", recap(conversation), len(conversation)))
else:
    print('Live example skipped; set JEV_LAB_LIVE=1 with your own API key to run.')

## 16 · The confidence gate — escalate only when jev isn't sure the cheap lane suffices

RouteLLM's cascade fails on miscalibrated self-confidence. jev's Noul is a **calibrated p(yes)**,
so we gate on the real probability: `p(cheap_answer_suffices)`. Above the band we trust the cheap
route; below it we escalate to the strong lane. First the raw probabilities on real follow-ups,
then the gated decision.

In [ ]:
if RUN_LIVE:
    from typesafe_sdk import TypeSafeClient, Noul

    def p_cheap_suffices(message: str, context_recap: str) -> float:
        state = f"Conversation so far:\n{context_recap}\n\nNew message:\n{message}"
        with TypeSafeClient() as client:
            r = client.system_one(state=state, questions={
                "cheap_ok": Noul(instructions="Can a cheap fast model answer this turn well given the conversation context, without a strong reasoning model?"),
            })
        return r.nouls["cheap_ok"].noul

    ctx = recap(conversation)
    for f in followups[:4]:
        print(f"p(cheap suffices)={p_cheap_suffices(f, ctx):.3f}  | {f[:60]}")
else:
    print('Live example skipped; set JEV_LAB_LIVE=1 with your own API key to run.')

## 16b · `decide_lane` — the per-turn decision over the data

Combines the classification (§15) with the confidence gate: a turn goes cheap only if jev *both*
classifies it cheap *and* is confident the cheap lane can answer. Otherwise it escalates.

In [ ]:
if RUN_LIVE:
    def decide_lane(new_message: str, context: list, confidence_threshold: float = 0.7) -> dict:
        ctx = recap(context)
        cls = route_turn(new_message, ctx, len(context))
        p = p_cheap_suffices(new_message, ctx)
        confident_cheap = p >= confidence_threshold
        if cls.lane == "cheap" and confident_cheap:
            lane, escalated = CHEAP_LANE, False
        else:
            lane, escalated = STRONG_LANE, (cls.lane == "cheap")   # classified cheap but not confident
        return {"lane": lane, "escalated": escalated, "p_cheap": round(p, 3),
                "followup": cls.followup_detected, "ctx_sufficient": cls.context_sufficient,
                "cost": LANE_COST[lane]}

    for f in followups[:4]:
        d = decide_lane(f, conversation)
        print(f"{d['lane'].split('/')[1]:<28} p={d['p_cheap']:<5} followup={d['followup']!s:<5} | {f[:50]}")
else:
    print('Live example skipped; set JEV_LAB_LIVE=1 with your own API key to run.')

## 17 · Run the whole thread — the follow-up discount, in cost

Now route the **actual conversation** turn by turn. The opener goes strong; the follow-ups — where
most of the tokens are — go cheap. Total the relative cost of **data-driven cognitive routing**
against the two baselines (every turn strong / every turn cheap). This is the CDR result reproduced
on your own conversation shape: the savings live in the follow-ups.

In [ ]:
if RUN_LIVE:
    def run_thread(opener, followups, confidence_threshold=0.7):
        ctx, decisions, total = [], [], 0.0
        # the opener
        d = decide_lane(opener, ctx, confidence_threshold)
        decisions.append((opener, d)); total += d["cost"]
        ctx.append({"role": "user", "text": opener})
        ctx.append({"role": "assistant", "text": f"[answered by {d['lane']}]"})
        # each follow-up rides the accumulated context
        for f in followups:
            d = decide_lane(f, ctx, confidence_threshold)
            decisions.append((f, d)); total += d["cost"]
            ctx.append({"role": "user", "text": f})
            ctx.append({"role": "assistant", "text": f"[answered by {d['lane']}]"})
        return decisions, total

    decisions, routed_cost = run_thread(conversation[0]["text"], followups)
    n_turns = len(decisions)
    always_strong = n_turns * LANE_COST[STRONG_LANE]
    always_cheap  = n_turns * LANE_COST[CHEAP_LANE]

    for i, (msg, d) in enumerate(decisions):
        tag = "OPENER " if i == 0 else f"follow{i}"
        print(f"{tag} cost={d['cost']:<3} p_cheap={d['p_cheap']:<5} escal={d['escalated']!s:<5} {d['lane'].split('/')[1]:<26} | {msg[:48]}")

    cheap_turns = sum(1 for _, d in decisions if d["lane"] == CHEAP_LANE)
    print(f"\n{n_turns} turns: {cheap_turns} cheap / {n_turns-cheap_turns} strong")
    print(f"relative cost — always-strong: {always_strong:.0f} | always-cheap: {always_cheap:.0f} | routed: {routed_cost:.1f}")
    print(f"savings vs always-strong: {100*(1 - routed_cost/always_strong):.0f}%  (the follow-up discount)")
else:
    print('Live example skipped; set JEV_LAB_LIVE=1 with your own API key to run.')

## Eighteen lenses
The bundled client supplies a deterministic calibration floor in `selected`; the raw scores remain model outputs. That floor must not be mistaken for a learned finding. No separate cognitive-agent checkout is needed.

In [ ]:
from jev_lab.lens_provider import JevDecisionProvider
from jev_lab.lenses import LENS_DEFINITIONS
print('Bundled lenses:', len(LENS_DEFINITIONS))
if RUN_LIVE:
    provider = JevDecisionProvider.from_env()

In [ ]:
if RUN_LIVE:
    turns = {
        "decision": "Should I spend $50K building this product now, or validate demand first?",
        "forecast": "Will this product reach $300K revenue within one year?",
        "sunk_cost": "We've already invested $2M, so we can't stop now — we have to see it through.",
        "factual": "What is the boiling point of water at sea level?",
    }

    results = {}
    for label, text in turns.items():
        d = await provider.select_lenses(text, LENS_DEFINITIONS, threshold=0.5, max_lenses=8)
        results[label] = d
        print(f"[{label}] selected {len(d.selected)} lenses:")
        print("   ", ", ".join(d.selected))
    print("\nmodel used:", results['decision'].model, "| request id:", results['decision'].request_id)
else:
    print('Live example skipped; set JEV_LAB_LIVE=1 with your own API key to run.')

In [ ]:
if RUN_LIVE:
    # (a) threshold as a cost dial on the sunk-cost turn
    text = turns["sunk_cost"]
    for thr in (0.3, 0.5, 0.7, 0.9):
        d = await provider.select_lenses(text, LENS_DEFINITIONS, threshold=thr, max_lenses=18)
        print(f"threshold={thr}: {len(d.selected):2d} lenses -> {', '.join(d.selected)}")

    # (b) probability mass per turn (top-5 lenses by p)
    print("\ntop lenses by calibrated probability:")
    for label, d in results.items():
        top = sorted(d.probabilities.items(), key=lambda kv: -kv[1])[:5]
        print(f"  [{label:9}] " + ", ".join(f"{k}={v:.2f}" for k, v in top))
else:
    print('Live example skipped; set JEV_LAB_LIVE=1 with your own API key to run.')

In [ ]:
if RUN_LIVE:
    print("lens-count per turn (threshold=0.5, the cheap-attention decision):")
    for label, d in results.items():
        bar = "#" * len(d.selected)
        print(f"  {label:10} {len(d.selected):2d} {bar}")

    # A factual question should select ~just calibration; a loaded decision should select several.
    factual_n = len(results["factual"].selected)
    decision_n = len(results["decision"].selected)
    print(f"\nfactual uses {factual_n} lens(es) vs decision {decision_n} -> "
          f"{'Jev allocates attention proportionally' if factual_n < decision_n else 'check thresholds'}")
else:
    print('Live example skipped; set JEV_LAB_LIVE=1 with your own API key to run.')

In [ ]:
if RUN_LIVE:
    d = results["decision"]
    print(d.prompt_context(threshold=0.5))
    print("\n--- this string is injected into the strong model's CognitiveAnalysisSignature ---")
    print("--- as `decision_context`: typed lens-selection cues, explicitly 'not evidence or findings' ---")
else:
    print('Live example skipped; set JEV_LAB_LIVE=1 with your own API key to run.')